## TextCNN Baseline — From Scratch

In [ ]:
import re, torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import wandb
from kaggle_secrets import UserSecretsClient

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## Clean & Encode

In [ ]:
def clean(t):
    return re.sub(r'[^a-z0-9 ]', '', t.lower())

# Vocab from training prompts
all_text = ' '.join(train['prompt'].apply(clean))
vocab = ['<PAD>','<UNK>'] + list(set(all_text.split()))
w2i = {w:i for i,w in enumerate(vocab)}

def encode(text, maxlen=50):
    ids = [w2i.get(w,1) for w in clean(text).split()][:maxlen]
    return ids + [0]*(maxlen-len(ids))

X = torch.tensor([encode(t) for t in train['prompt']], dtype=torch.long)
y = torch.tensor([ord(a)-ord('A') for a in train['answer']], dtype=torch.long)

print("X:", X.shape, "| y:", y.shape, "| Vocab:", len(vocab))

## Model

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.conv = nn.Conv1d(64, 64, kernel_size=3)
        self.fc = nn.Linear(64, 5)   # 5 classes: A B C D E

    def forward(self, x):
        x = self.emb(x).permute(0,2,1)
        x = F.relu(self.conv(x))
        x = F.max_pool1d(x, x.shape[2]).squeeze(2)
        return self.fc(x)

model = TextCNN(len(vocab))
print(model)

In [ ]:
def map_at3(predictions, correct):
    score = 0
    for pred, ans in zip(predictions, correct):
        if ans in pred:
            score += 1 / (pred.index(ans) + 1)
    return round(score / len(correct), 4)

## Train + WandB

In [ ]:
from sklearn.model_selection import train_test_split

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GENAI_T2_2026")

wandb.login(key=secret_value_0)
wandb.init(
    project = "smart-mcq-solver",
    name = "textcnn-baseline - v2",
    config  = {"vocab_size": len(vocab), "embed_dim": 64,
               "num_filters": 64, "kernel_size": 3,
               "lr": 1e-3, "epochs": 10}
)

# Val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)

options = ['A','B','C','D','E']
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    # Train
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train), y_train)
    loss.backward()
    optimizer.step()

    # Val MAP@3
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(X_val), dim=1).numpy()

    correct = [options[i] for i in y_val.numpy()]
    preds = [[options[j] for j in row.argsort()[::-1][:3]] for row in probs]
    score = map_at3(preds, correct)

    print(f"Epoch {epoch+1}/10 | Loss: {loss.item():.4f} | Val MAP@3: {score:.4f}")
    wandb.log({"epoch": epoch+1, "train_loss": loss.item(), "val_map3": score})

wandb.finish()

## Submit

In [ ]:
model.eval()
X_test = torch.tensor([encode(t) for t in test['prompt']], dtype=torch.long)

with torch.no_grad():
    probs = torch.softmax(model(X_test), dim=1).numpy()

options = ['A','B','C','D','E']
rows = []
for i, row in test.iterrows():
    top3 = ' '.join([options[j] for j in probs[i].argsort()[::-1][:3]])
    rows.append({'id': row['id'], 'prediction': top3})

pd.DataFrame(rows).to_csv('submission.csv', index=False)
print("Done")
print(pd.DataFrame(rows).head())